[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/11_generative_recsys/11_proyecto_semantic_ids.ipynb)

# Proyecto 11 · Semantic IDs (RQ-VAE) + un pequeño modelo generativo de *retrieval*

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración** | 4 h |
| **GPU** | A100 recomendada (L4 con menos épocas) |
| **Unidades Colab** | ≈ 5–8 (A100) |
| **Prerrequisitos** | Lección 11 |

## 🏢 Contexto de negocio
CineMatch va a lanzar una **tienda de videojuegos y merchandising** dentro de la app. El catálogo cambia cada semana (≈ 25 k productos, cientos de altas semanales) y el
two-tower actual sufre con los productos nuevos (no tienen fila en la tabla de embeddings). El equipo de investigación propone un piloto de **generative retrieval con semantic IDs**:
los productos nuevos reciben un ID a partir de su **texto**, sin reentrenar el recomendador.

Tu misión: construir el tokenizador (RQ-VAE → semantic IDs), el modelo generativo (encoder-decoder) con decodificación restringida, y una comparación honesta con popularidad y con SASRec.

## 📦 Dataset
**Amazon Reviews 2023** (McAuley Lab), categoría `Video_Games`, *benchmark* 5-core (`McAuley-Lab/Amazon-Reviews-2023` en Hugging Face) + metadatos de producto.
Protocolo *leave-one-out*. Fallback automático a MovieLens-1M si no hay acceso a Hugging Face.

## ✅ Entregables y rúbrica
| Criterio | Objetivo |
|---|---|
| RQ-VAE: uso del codebook de nivel 1 | ≥ 90 % de los 256 códigos usados |
| Colisiones antes del token de desambiguación | < 15 % de los ítems |
| Pureza de categoría de nivel 1 | claramente > asignación aleatoria |
| Beam search con trie | 0 % de IDs inválidos |
| Recall@10 del modelo generativo en test | ≥ 2× popularidad; reporta la distancia a SASRec y analiza por popularidad del ítem |
| Experimento de *cold start* (reto guiado) | Recall@10 sobre ítems retirados del entrenamiento > 0 |

In [ ]:
# sentence-transformers (embeddings de texto de ítems) y huggingface_hub (Amazon Reviews 2023)
!pip install -q sentence-transformers huggingface_hub

In [ ]:
import os, json, math, time, random, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = device == "cuda" and torch.cuda.is_bf16_supported()
FAST_DEV_RUN = False          # True = iterar rápido; False = experimento completo (A100/L4)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("device:", device, "| bf16:", USE_AMP, "| torch", torch.__version__)

## 🧰 Utilidades mínimas (vienen de los módulos 01 y 02)

Para que este notebook sea **autocontenido en Colab**, copiamos aquí una versión reducida de las utilidades que construimos en
`01_data` (carga de MovieLens, secuencias, *splits*) y `02_evaluation` (HR@K y NDCG@K con *full ranking*).

- Si la descarga falla (sin red, *firewall*…), se generan **datos sintéticos con estructura secuencial** para que el notebook nunca se bloquee. Los números con datos sintéticos **no** son comparables con los papers.
- Convención de IDs: los ítems se re-indexan a `1..n_items` y **el 0 es el *padding***.

In [ ]:
# ==== Utilidades (versión mínima de 01_data) ====
import os, io, zipfile, urllib.request

ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"

def make_synthetic_ml(n_users=1500, n_items=700, n_genres=12, seed=42):
    '''Datos de juguete con señal secuencial: "siguiente capítulo", deriva de género y popularidad.'''
    rng = np.random.default_rng(seed)
    genre = rng.integers(0, n_genres, n_items)
    by_genre = [np.where(genre == g)[0] for g in range(n_genres)]
    pop = np.minimum(rng.zipf(1.4, n_items), 60).astype(float); pop /= pop.sum()
    rows = []
    for u in range(1, n_users + 1):
        g = int(rng.integers(n_genres)); it = int(rng.choice(by_genre[g])); t = 978_300_000 + int(rng.integers(0, 10**7))
        for _ in range(int(rng.integers(20, 160))):
            rows.append((u, it + 1, int(rng.integers(3, 6)), t))
            t += int(rng.integers(30, 3 * 86400)); r = rng.random()
            if r < 0.45:   it = (it + int(rng.integers(1, 4))) % n_items          # secuela / siguiente capítulo
            elif r < 0.8:  it = int(rng.choice(by_genre[g]))                       # mismo género
            else:          g = int(rng.integers(n_genres)); it = int(rng.choice(n_items, p=pop))
    ratings = pd.DataFrame(rows, columns=["user", "item", "rating", "ts"]).drop_duplicates(["user", "item"])
    names = ["Action", "Comedy", "Drama", "Horror", "Sci-Fi", "Romance", "Thriller",
             "Animation", "Documentary", "Crime", "Musical", "War"]
    movies = pd.DataFrame({"item": np.arange(1, n_items + 1),
                           "title": [f"Película sintética {i} (2000)" for i in range(1, n_items + 1)],
                           "genres": [names[g % len(names)] for g in genre]})
    return ratings, movies

def load_movielens_1m(data_dir="data"):
    '''Devuelve (ratings[user,item,rating,ts], movies[item,title,genres]).'''
    os.makedirs(data_dir, exist_ok=True)
    zpath = os.path.join(data_dir, "ml-1m.zip")
    try:
        if not os.path.exists(zpath):
            urllib.request.urlretrieve(ML1M_URL, zpath)
        with zipfile.ZipFile(zpath) as z:
            ratings = pd.read_csv(z.open("ml-1m/ratings.dat"), sep="::", engine="python",
                                  names=["user", "item", "rating", "ts"])
            movies = pd.read_csv(z.open("ml-1m/movies.dat"), sep="::", engine="python",
                                 names=["item", "title", "genres"], encoding="latin-1")
        return ratings, movies
    except Exception as e:
        if os.path.exists(zpath):
            os.remove(zpath)
        print(f"⚠️ No se pudo descargar MovieLens-1M ({type(e).__name__}). Uso datos SINTÉTICOS.")
        return make_synthetic_ml()

In [ ]:
# ==== Secuencias y split leave-one-out (01_data) ====
def build_sequences(ratings, min_len=5):
    '''Ordena por tiempo y devuelve (seqs: list[list[int]], item2idx, idx2item). 0 = padding.'''
    df = ratings.sort_values(["user", "ts"], kind="mergesort")      # estable: respeta el orden del fichero en empates
    cnt = df.groupby("user")["item"].transform("size")
    df = df[cnt >= min_len]
    items = np.sort(df["item"].unique())
    item2idx = {it: i + 1 for i, it in enumerate(items)}
    idx2item = np.concatenate([[0], items])
    df = df.assign(iidx=df["item"].map(item2idx))
    seqs = df.groupby("user", sort=True)["iidx"].apply(list).tolist()
    return seqs, item2idx, idx2item

def leave_one_out(seqs):
    '''train = s[:-2]; valid: (s[:-2] -> s[-2]); test: (s[:-1] -> s[-1]).'''
    train = [s[:-2] for s in seqs]
    valid = ([s[:-2] for s in seqs], [s[-2] for s in seqs])
    test = ([s[:-1] for s in seqs], [s[-1] for s in seqs])
    return train, valid, test

def pad_left(seqs, maxlen):
    '''Trunca a los últimos `maxlen` y rellena con 0 por la izquierda -> LongTensor [B, maxlen].'''
    out = np.zeros((len(seqs), maxlen), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = s[-maxlen:]
        if len(s):
            out[i, -len(s):] = s
    return torch.from_numpy(out)

In [ ]:
# ==== Métricas HR@K / NDCG@K con full ranking (02_evaluation) ====
@torch.no_grad()
def evaluate_next_item(score_fn, histories, targets, ks=(10,), batch_size=512, mask_history=True):
    '''score_fn(list[list[int]]) -> Tensor [B, n_items+1] con puntuaciones de TODOS los ítems.
    Ranking completo (sin negativos muestreados, ver Krichene & Rendle 2020). Una sola relevante => HR = Recall.'''
    ranks = []
    for b in range(0, len(histories), batch_size):
        H, T = histories[b:b + batch_size], torch.as_tensor(targets[b:b + batch_size])
        s = score_fn(H).float().cpu()
        tgt = s.gather(1, T[:, None]).clone()
        s[:, 0] = -float("inf")                                # padding
        if mask_history:                                       # no recomendar lo ya visto
            for i, h in enumerate(H):
                s[i, h[0] if isinstance(h, tuple) else h] = -float("inf")   # h puede ser (items, timestamps)
        s.scatter_(1, T[:, None], tgt)                         # el objetivo nunca se enmascara
        ranks.append((s > tgt).sum(1))                         # posición 0-based (empates optimistas)
    ranks = torch.cat(ranks).numpy()
    out = {}
    for k in ks:
        hit = ranks < k
        out[f"HR@{k}"] = float(hit.mean())
        out[f"NDCG@{k}"] = float((hit / np.log2(ranks + 2)).mean())
    return out

In [ ]:
AMZ_CATEGORY = "Video_Games"     # otras opciones: "Industrial_and_Scientific", "Musical_Instruments", "Toys_and_Games"...

def load_amazon2023(cat=AMZ_CATEGORY):
    '''Interacciones 5-core (benchmark oficial) + metadatos de Amazon Reviews 2023 (McAuley Lab) desde Hugging Face.'''
    from huggingface_hub import hf_hub_download
    repo = "McAuley-Lab/Amazon-Reviews-2023"
    inter = pd.read_csv(hf_hub_download(repo, f"benchmark/5core/rating_only/{cat}.csv", repo_type="dataset"))
    keep, rows = set(inter["parent_asin"]), []
    with open(hf_hub_download(repo, f"raw/meta_categories/meta_{cat}.jsonl", repo_type="dataset")) as f:
        for line in f:
            d = json.loads(line)
            if d.get("parent_asin") not in keep:
                continue
            cats = d.get("categories") or []
            text = (f"Title: {d.get('title') or ''}. Store: {d.get('store') or ''}. Categories: {', '.join(cats)}. "
                    f"Price: {d.get('price')}. Features: {' '.join((d.get('features') or [])[:3])}")[:1000]
            rows.append({"item": d["parent_asin"], "title": d.get("title") or "", "text": text,
                         "cat": cats[1] if len(cats) > 1 else (cats[0] if cats else "?")})
    meta = pd.DataFrame(rows).drop_duplicates("item")
    r = inter.rename(columns={"user_id": "user", "parent_asin": "item", "timestamp": "ts"})
    return r[r["item"].isin(set(meta["item"]))][["user", "item", "rating", "ts"]], meta

def load_generative_dataset():
    try:
        r, meta = load_amazon2023(); src = f"Amazon Reviews 2023 · {AMZ_CATEGORY} (5-core)"
    except Exception as e:
        print(f"⚠️ Amazon Reviews 2023 no disponible ({type(e).__name__}) → uso MovieLens-1M (título + géneros como texto)")
        r, mv = load_movielens_1m()
        meta = pd.DataFrame({"item": mv["item"], "title": mv["title"], "cat": mv["genres"].str.split("|").str[0],
                             "text": "Title: " + mv["title"] + ". Genres: " + mv["genres"].str.replace("|", ", ")})
        src = "MovieLens-1M"
    return r, meta, src

g_ratings, g_meta, g_src = load_generative_dataset()
g_seqs, g_item2idx, g_idx2item = build_sequences(g_ratings, min_len=5)
gn_items = len(g_item2idx)
g_train, g_valid, g_test = leave_one_out(g_seqs)
meta_idx = g_meta.set_index("item")
item_text = [meta_idx.loc[g_idx2item[i], "text"] for i in range(1, gn_items + 1)]
item_title = ["<pad>"] + [str(meta_idx.loc[g_idx2item[i], "title"])[:40] for i in range(1, gn_items + 1)]
item_cat = np.array(["<pad>"] + [str(meta_idx.loc[g_idx2item[i], "cat"]) for i in range(1, gn_items + 1)])
lens = np.array([len(s) for s in g_seqs])
print(f"{g_src}: usuarios={len(g_seqs):,} ítems={gn_items:,} interacciones={lens.sum():,} long. media={lens.mean():.1f}")
print("Ejemplo de texto de ítem:", item_text[0][:200])

In [ ]:
def embed_texts(texts, model_name="sentence-transformers/sentence-t5-base", cache=None):
    '''Embeddings de contenido (TIGER usó Sentence-T5). Fallback offline: TF-IDF + SVD.'''
    if cache and os.path.exists(cache):
        X = np.load(cache)
        if X.shape[0] == len(texts):
            return X
    try:
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer(model_name, device=device)
        X = st.encode(texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)
    except Exception as e:
        cache = None                                                # nunca cacheamos el fallback
        print(f"⚠️ sentence-transformers no disponible ({type(e).__name__}) → TF-IDF + SVD")
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        T = TfidfVectorizer(min_df=1, max_features=50000, ngram_range=(1, 2)).fit_transform(texts)
        X = TruncatedSVD(min(256, T.shape[1] - 1), random_state=seed).fit_transform(T)
        X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    X = X.astype(np.float32)
    if cache:
        os.makedirs(os.path.dirname(cache), exist_ok=True); np.save(cache, X)
    return X

X_items = embed_texts(item_text, cache=f"data/emb_{g_src.split(' ')[0]}_{gn_items}.npy")
print("embeddings de ítems:", X_items.shape)

## Paso 1 · Cuantización residual
Completa `quantize`: para cada *codebook*, código más cercano al residuo, pérdida de codebook + β·commitment, acumula $\hat z$ y actualiza el residuo.
El resto del RQ-VAE (encoder/decoder, STE, k-means) está dado.

In [ ]:
def kmeans_torch(x, K, iters=25, seed=seed):
    g = torch.Generator(device=x.device).manual_seed(seed)
    idx = torch.randperm(len(x), generator=g, device=x.device)[:K] if len(x) >= K else torch.randint(len(x), (K,), generator=g, device=x.device)
    C = x[idx].clone()
    for _ in range(iters):
        a = torch.cdist(x, C).argmin(1)
        S = torch.zeros_like(C).index_add_(0, a, x); cnt = torch.bincount(a, minlength=K).float()
        C = torch.where((cnt == 0)[:, None], x[torch.randint(len(x), (K,), generator=g, device=x.device)], S / cnt.clamp(min=1)[:, None])
    return C, torch.cdist(x, C).argmin(1)

class RQVAE(nn.Module):
    def __init__(self, in_dim, latent=32, hidden=(512, 256, 128), levels=3, K=256, beta=0.25):
        super().__init__()
        dims, enc, dec = [in_dim, *hidden], [], []
        for a, b in zip(dims[:-1], dims[1:]): enc += [nn.Linear(a, b), nn.ReLU()]
        self.encoder = nn.Sequential(*enc, nn.Linear(dims[-1], latent))
        rd = [latent, *dims[::-1]]
        for a, b in zip(rd[:-1], rd[1:]): dec += [nn.Linear(a, b), nn.ReLU()]
        self.decoder = nn.Sequential(*dec[:-1])
        self.codebooks = nn.ParameterList([nn.Parameter(0.1 * torch.randn(K, latent)) for _ in range(levels)])
        self.K, self.L, self.beta = K, levels, beta

    def quantize(self, z):
        # TODO: devuelve (z_hat [B,d], codes [B,L], cb_loss escalar)
        raise NotImplementedError

    def forward(self, x):
        z = self.encoder(x); z_hat, codes, cb = self.quantize(z)
        x_hat = self.decoder(z + (z_hat - z).detach())
        return x_hat, codes, ((x_hat - x) ** 2).sum(1).mean(), cb

    @torch.no_grad()
    def init_codebooks(self, x):
        r = self.encoder(x)
        for C in self.codebooks:
            cent, a = kmeans_torch(r, self.K); C.data.copy_(cent); r = r - cent[a]

    @torch.no_grad()
    def get_codes(self, x, batch=8192):
        return torch.cat([self.quantize(self.encoder(x[b:b + batch]))[1] for b in range(0, len(x), batch)])

## Paso 2 · Entrenar el tokenizador y diagnosticarlo
Escribe el bucle de entrenamiento (Adam, *batches* de 1024, ~300–400 épocas en GPU), con **reinicio de códigos muertos** cada 20 épocas, y reporta: uso por nivel, % de colisiones y pureza de categoría del nivel 1 vs aleatorio.

In [ ]:
def train_rqvae(X, K=256, levels=3, latent=32, epochs=300, lr=1e-3, batch=1024, reset_every=20, log_every=25):
    # TODO: devuelve (modelo, historial)
    raise NotImplementedError

## Paso 3 · IDs únicos y trie
1. Añade el token de desambiguación. 2. Implementa `allowed_mask(prefixes, level, size)` del trie (puedes empezar con la versión de diccionario y luego vectorizarla).

In [ ]:
def add_dedup_token(codes):
    # TODO
    raise NotImplementedError

class SemanticIDTrie:
    def __init__(self, codes):
        # TODO: estructura que permita (a) hijos válidos de un prefijo y (b) traducir un ID completo a índice de ítem
        raise NotImplementedError
    def allowed_mask(self, prefixes, level, size):
        raise NotImplementedError
    def lookup(self, codes):
        raise NotImplementedError

## Paso 4 · Modelo generativo y beam search restringido
Se da la clase `TigerMini` (lección). Implementa `beam_search` con el trie y el bucle de entrenamiento; evalúa Recall@10/NDCG@10 con beam = 20.

In [ ]:
class TigerMini(nn.Module):
    '''Encoder-decoder (estilo TIGER): historial como tokens de semantic IDs -> genera el ID del siguiente ítem.'''
    def __init__(self, vocab, level_sizes, offsets, d=128, nhead=4, n_layers=4, ff=1024, dropout=0.1, max_src=80):
        super().__init__()
        self.L, self.vocab, self.bos = len(level_sizes), vocab, vocab - 1
        self.level_sizes, self.offsets = list(level_sizes), list(offsets)
        self.tok = nn.Embedding(vocab, d, padding_idx=0)
        self.src_pos, self.tgt_pos = nn.Embedding(max_src, d), nn.Embedding(self.L + 1, d)
        self.tf = nn.Transformer(d_model=d, nhead=nhead, num_encoder_layers=n_layers, num_decoder_layers=n_layers,
                                 dim_feedforward=ff, dropout=dropout, batch_first=True, norm_first=True)
        self.head = nn.Linear(d, vocab)
        lm = torch.full((self.L, vocab), float("-inf"))
        for l, (o, s) in enumerate(zip(offsets, level_sizes)):
            lm[l, o:o + s] = 0.0                                   # la posición l sólo emite tokens del nivel l
        self.register_buffer("level_mask", lm)

    def encode(self, src):
        pad = src == 0
        x = self.tok(src) + self.src_pos(torch.arange(src.size(1), device=src.device))
        return self.tf.encoder(x, src_key_padding_mask=pad), pad

    def decode(self, tgt_in, memory, pad):
        T = tgt_in.size(1)
        y = self.tok(tgt_in) + self.tgt_pos(torch.arange(T, device=tgt_in.device))
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), 1)
        out = self.tf.decoder(y, memory, tgt_mask=causal, memory_key_padding_mask=pad)
        return self.head(out) + self.level_mask[:T]

    def forward(self, src, tgt):                                    # tgt [B, L] tokens con offset
        memory, pad = self.encode(src)
        tgt_in = torch.cat([torch.full_like(tgt[:, :1], self.bos), tgt[:, :-1]], 1)
        logits = self.decode(tgt_in, memory, pad)
        return F.cross_entropy(logits.reshape(-1, self.vocab).float(), tgt.reshape(-1))

In [ ]:
@torch.no_grad()
def beam_search(model, src, trie, beam=20, constrained=True):
    # TODO: devuelve (codes [B, beam, L], scores [B, beam])
    raise NotImplementedError

## Paso 5 · *Cold start* (reto guiado)
Retira del entrenamiento del modelo generativo el 5 % de los ítems (todas sus apariciones como objetivo **y** en historiales), pero mantenlos en el tokenizador y en el trie.
Mide el Recall@10 en los usuarios de test cuyo objetivo es uno de esos ítems. Un modelo con IDs aleatorios (SASRec) tendría Recall@10 = 0 en ellos.

In [ ]:
# TODO: experimento de cold start
...

---
# ⛔ SPOILER — Solución de referencia

In [ ]:
def kmeans_torch(x, K, iters=25, seed=seed):
    '''k-means sencillo en GPU. Devuelve (centroides [K,d], asignación [N]).'''
    g = torch.Generator(device=x.device).manual_seed(seed)
    idx = (torch.randperm(len(x), generator=g, device=x.device)[:K] if len(x) >= K
           else torch.randint(len(x), (K,), generator=g, device=x.device))
    C = x[idx].clone() + (1e-4 * torch.randn(K, x.size(1), generator=g, device=x.device) if len(x) < K else 0)
    for _ in range(iters):
        a = torch.cdist(x, C).argmin(1)
        S = torch.zeros_like(C).index_add_(0, a, x); cnt = torch.bincount(a, minlength=K).float()
        refill = x[torch.randint(len(x), (K,), generator=g, device=x.device)]
        C = torch.where((cnt == 0)[:, None], refill, S / cnt.clamp(min=1)[:, None])
    return C, torch.cdist(x, C).argmin(1)


class RQVAE(nn.Module):
    '''RQ-VAE (TIGER): encoder MLP -> cuantización residual con L codebooks de K códigos -> decoder MLP.'''
    def __init__(self, in_dim, latent=32, hidden=(512, 256, 128), levels=3, K=256, beta=0.25):
        super().__init__()
        dims, enc, dec = [in_dim, *hidden], [], []
        for a, b in zip(dims[:-1], dims[1:]):
            enc += [nn.Linear(a, b), nn.ReLU()]
        self.encoder = nn.Sequential(*enc, nn.Linear(dims[-1], latent))
        rd = [latent, *dims[::-1]]
        for a, b in zip(rd[:-1], rd[1:]):
            dec += [nn.Linear(a, b), nn.ReLU()]
        self.decoder = nn.Sequential(*dec[:-1])                       # sin ReLU final
        self.codebooks = nn.ParameterList([nn.Parameter(0.1 * torch.randn(K, latent)) for _ in range(levels)])
        self.K, self.L, self.beta = K, levels, beta

    def quantize(self, z):
        r, z_hat, codes, cb_loss = z, torch.zeros_like(z), [], 0.0
        for C in self.codebooks:
            c = torch.cdist(r, C).argmin(1); e = C[c]                     # código más cercano al residuo
            cb_loss = cb_loss + ((e - r.detach()) ** 2).sum(1).mean() + self.beta * ((r - e.detach()) ** 2).sum(1).mean()
            z_hat, r = z_hat + e, r - e.detach()                          # siguiente residuo
            codes.append(c)
        return z_hat, torch.stack(codes, 1), cb_loss

    def forward(self, x):
        z = self.encoder(x)
        z_hat, codes, cb_loss = self.quantize(z)
        x_hat = self.decoder(z + (z_hat - z).detach())                    # straight-through estimator
        return x_hat, codes, ((x_hat - x) ** 2).sum(1).mean(), cb_loss

    @torch.no_grad()
    def init_codebooks(self, x):                                          # k-means por nivel (truco de TIGER)
        r = self.encoder(x)
        for C in self.codebooks:
            cent, a = kmeans_torch(r, self.K); C.data.copy_(cent); r = r - cent[a]

    @torch.no_grad()
    def get_codes(self, x, batch=8192):
        return torch.cat([self.quantize(self.encoder(x[b:b + batch]))[1] for b in range(0, len(x), batch)])

In [ ]:
def collision_rate(codes):
    return 1 - len(np.unique(codes, axis=0)) / len(codes)

def add_dedup_token(codes):
    '''TIGER: añade un último token = índice del ítem dentro de su grupo de colisión -> IDs únicos.'''
    dup = pd.DataFrame(codes).groupby(list(range(codes.shape[1]))).cumcount().values
    return np.concatenate([codes, dup[:, None]], 1)

def train_rqvae(X, K=256, levels=3, latent=32, epochs=300, lr=1e-3, batch=1024, reset_every=20, log_every=25):
    Xt = torch.from_numpy(X).to(device)
    m = RQVAE(X.shape[1], latent, levels=levels, K=K).to(device)
    m.init_codebooks(Xt[torch.randperm(len(Xt), device=device)[:20000]])
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    hist = collections.defaultdict(list)
    for ep in range(1, epochs + 1):
        m.train(); perm = torch.randperm(len(Xt), device=device)
        usage = torch.zeros(levels, K, device=device); rec_t = cb_t = 0.0
        for b in range(0, len(Xt), batch):
            x = Xt[perm[b:b + batch]]
            _, codes, rec, cb = m(x)
            loss = rec + cb
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            for l in range(levels):
                usage[l].index_add_(0, codes[:, l], torch.ones(len(codes), device=device))
            rec_t += rec.item() * len(x); cb_t += cb.item() * len(x)
        if reset_every and ep % reset_every == 0 and ep < epochs:          # reinicia códigos muertos
            with torch.no_grad():
                r = m.encoder(Xt[torch.randint(len(Xt), (4096,), device=device)])
                for l, C in enumerate(m.codebooks):
                    dead = usage[l] == 0
                    if dead.any():
                        C.data[dead] = r[torch.randint(len(r), (int(dead.sum()),), device=device)]
                    r = r - C[torch.cdist(r, C).argmin(1)]
        if ep % log_every == 0 or ep == epochs:
            codes_all = m.get_codes(Xt).cpu().numpy()
            hist["epoch"].append(ep); hist["recon"].append(rec_t / len(Xt)); hist["codebook"].append(cb_t / len(Xt))
            hist["collisions"].append(collision_rate(codes_all))
            for l in range(levels):
                hist[f"uso_nivel{l+1}"].append(len(np.unique(codes_all[:, l])) / K)
            print(f"ep {ep:4d} recon={hist['recon'][-1]:.4f} cb={hist['codebook'][-1]:.4f} "
                  f"colisiones={hist['collisions'][-1]:.2%} uso={[round(hist[f'uso_nivel{l+1}'][-1], 2) for l in range(levels)]}")
    return m, dict(hist)

In [ ]:
class SemanticIDTrie:
    '''Árbol de prefijos de semantic IDs. Versión didáctica (dict) + versión vectorizada en GPU (searchsorted).'''
    def __init__(self, codes):                                   # codes [N, L] (sin offsets), fila i -> ítem i+1
        self.L, self.base = codes.shape[1], int(codes.max()) + 2
        self.children = collections.defaultdict(set)
        for c in codes[: min(len(codes), 200_000)]:
            for l in range(self.L):
                self.children[tuple(int(v) for v in c[:l])].add(int(c[l]))
        self.keys, self.nexts = [], []
        for l in range(self.L):
            pairs = np.unique(np.stack([self._key_np(codes[:, :l]), codes[:, l]], 1), axis=0)   # ordenado por clave
            self.keys.append(torch.from_numpy(pairs[:, 0]).to(device)); self.nexts.append(torch.from_numpy(pairs[:, 1]).to(device))
        full = self._key_np(codes)
        order = np.argsort(full)
        self.item_keys = torch.from_numpy(full[order]).to(device)
        self.item_ids = torch.from_numpy(order + 1).to(device)

    def _key_np(self, P):
        k = np.zeros(len(P), dtype=np.int64)
        for j in range(P.shape[1]):
            k = k * self.base + P[:, j] + 1
        return k

    def _key(self, P):
        k = torch.zeros(P.size(0), dtype=torch.long, device=P.device)
        for j in range(P.size(1)):
            k = k * self.base + P[:, j] + 1
        return k

    def allowed_mask(self, prefixes, level, size):
        '''prefixes [n, level] -> máscara bool [n, size] con los siguientes códigos válidos.'''
        q = self._key(prefixes)
        lo = torch.searchsorted(self.keys[level], q); hi = torch.searchsorted(self.keys[level], q, right=True)
        cnt = hi - lo
        rows = torch.repeat_interleave(torch.arange(len(q), device=q.device), cnt)
        offs = torch.arange(int(cnt.sum()), device=q.device) - torch.repeat_interleave(torch.cumsum(cnt, 0) - cnt, cnt)
        mask = torch.zeros(len(q), size, dtype=torch.bool, device=q.device)
        mask[rows, self.nexts[level][lo[rows] + offs]] = True
        return mask

    def lookup(self, codes):
        '''codes [..., L] -> índice de ítem (0 si el ID no existe).'''
        flat = codes.reshape(-1, self.L); k = self._key(flat)
        pos = torch.searchsorted(self.item_keys, k).clamp(max=len(self.item_keys) - 1)
        found = self.item_keys[pos] == k
        return torch.where(found, self.item_ids[pos], torch.zeros_like(pos)).view(codes.shape[:-1])

In [ ]:
def build_token_tables(sid_codes, K):
    '''sid_codes [n_items, L] -> tabla item -> tokens (con offset por nivel; 0 = padding).'''
    level_sizes = [K] * (sid_codes.shape[1] - 1) + [int(sid_codes[:, -1].max()) + 1]
    offsets = (1 + np.concatenate([[0], np.cumsum(level_sizes)[:-1]])).astype(np.int64)
    vocab = 1 + int(sum(level_sizes)) + 1                          # + PAD + BOS
    tok = np.zeros((len(sid_codes) + 1, sid_codes.shape[1]), dtype=np.int64)
    tok[1:] = sid_codes + offsets
    return torch.from_numpy(tok), level_sizes, offsets, vocab

def make_src(histories, item_tokens, H=20):
    items = pad_left(histories, H)                                  # [B, H]
    return item_tokens[items].reshape(len(histories), -1)          # [B, H*L]; el padding (ítem 0) -> tokens 0

def gen_examples(train_seqs, H=20, max_examples=None, rng=np.random.default_rng(seed)):
    hist, tgt = [], []
    for s in train_seqs:
        for t in range(1, len(s)):
            hist.append(s[max(0, t - H):t]); tgt.append(s[t])
    if max_examples and len(tgt) > max_examples:
        sel = rng.choice(len(tgt), max_examples, replace=False)
        hist, tgt = [hist[i] for i in sel], [tgt[i] for i in sel]
    return hist, np.array(tgt)

def train_tiger(model, hist, tgt, item_tokens, epochs=10, batch_size=512, lr=5e-4, H=20, val=None, eval_fn=None):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * math.ceil(len(tgt) / batch_size)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=total, pct_start=0.05)
    log = collections.defaultdict(list); t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); perm = np.random.permutation(len(tgt)); tot = 0.0
        for b in range(0, len(perm), batch_size):
            idx = perm[b:b + batch_size]
            src = make_src([hist[i] for i in idx], item_tokens, H).to(device)
            y = item_tokens[tgt[idx]].to(device)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_AMP):
                loss = model(src, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tot += loss.item() * len(idx)
        log["epoch"].append(ep); log["loss"].append(tot / len(tgt))
        msg = f"ep {ep:2d} loss={tot/len(tgt):.4f} ({time.time()-t0:.0f}s)"
        if eval_fn is not None:
            r = eval_fn(model); log["val_Recall@10"].append(r["Recall@10"]); msg += f"  val Recall@10={r['Recall@10']:.4f}"
        print(msg)
    return dict(log)

In [ ]:
@torch.no_grad()
def beam_search(model, src, trie, beam=20, constrained=True):
    '''Beam search autoregresivo sobre semantic IDs. Devuelve códigos [B, beam, L] (sin offset) y log-probs [B, beam].'''
    model.eval()
    B = src.size(0)
    memory, pad = model.encode(src)
    codes = torch.zeros(B, 1, 0, dtype=torch.long, device=src.device)
    scores = torch.zeros(B, 1, device=src.device)
    off = torch.tensor(model.offsets, device=src.device)
    for l in range(model.L):
        nb, size = codes.size(1), model.level_sizes[l]
        prev_tok = codes + off[:l] if l > 0 else codes
        tgt_in = torch.cat([torch.full((B, nb, 1), model.bos, device=src.device), prev_tok], 2).view(B * nb, l + 1)
        logits = model.decode(tgt_in, memory.repeat_interleave(nb, 0), pad.repeat_interleave(nb, 0))[:, -1]
        logp = logits[:, model.offsets[l]:model.offsets[l] + size].float().log_softmax(-1)      # [B*nb, size]
        if constrained:                                                 # sólo continuaciones que existen en el trie
            logp = logp.masked_fill(~trie.allowed_mask(codes.view(B * nb, l), l, size), float("-inf"))
        cand = (scores.view(B * nb, 1) + logp).view(B, nb * size)
        top_s, top_i = cand.topk(min(beam, cand.size(1)), 1)
        bidx, tok = top_i // size, top_i % size
        codes = torch.cat([codes.gather(1, bidx[..., None].expand(-1, -1, l)), tok[..., None]], 2)
        scores = top_s
    return codes, scores

@torch.no_grad()
def eval_generative(model, histories, targets, trie, item_tokens, beam=20, ks=(10, 20), H=20, batch_size=256,
                    constrained=True, return_ranks=False):
    ranks, invalid, total = [], 0, 0
    for b in range(0, len(histories), batch_size):
        Hs = histories[b:b + batch_size]
        codes, scores = beam_search(model, make_src(Hs, item_tokens, H).to(device), trie, beam, constrained)
        items = trie.lookup(codes)                                       # [B, beam] (0 = ID inexistente)
        valid = (items > 0) & torch.isfinite(scores)
        invalid += int((~valid).sum()); total += valid.numel()
        tg = torch.as_tensor(targets[b:b + batch_size], device=items.device)
        for i, h in enumerate(Hs):                                       # no recomendar lo ya visto (igual que en 09:
            seen = torch.isin(items[i], torch.as_tensor(h[-200:], device=items.device))   # el objetivo nunca se enmascara)
            valid[i] &= ~(seen & (items[i] != tg[i]))
        rank_in_valid = valid.long().cumsum(1) - 1                       # posición tras filtrar
        hit = valid & (items == tg[:, None])
        r = torch.where(hit.any(1), (rank_in_valid * hit).sum(1), torch.full_like(hit[:, 0], 10**6, dtype=torch.long))
        ranks.append(r.cpu())
    ranks = torch.cat(ranks).numpy()
    out = {"invalid_rate": invalid / total}
    for k in ks:
        out[f"Recall@{k}"] = float((ranks < k).mean()); out[f"NDCG@{k}"] = float(((ranks < k) / np.log2(ranks + 2)).mean())
    return (out, ranks) if return_ranks else out

In [ ]:
CODEBOOK_SIZE = 256
torch.manual_seed(seed)
rqvae, rq_hist = train_rqvae(X_items, K=CODEBOOK_SIZE, epochs=30 if FAST_DEV_RUN else 400, log_every=10 if FAST_DEV_RUN else 50)
codes3 = rqvae.get_codes(torch.from_numpy(X_items).to(device)).cpu().numpy()
cats = item_cat[1:]
def purity(c1, cats, min_size=5):
    g = pd.DataFrame({"c": c1, "cat": cats}).groupby("c")["cat"].agg(lambda s: s.value_counts(normalize=True).iloc[0] if len(s) >= min_size else np.nan)
    return g.dropna().values
print(f"uso nivel 1: {len(np.unique(codes3[:, 0])) / CODEBOOK_SIZE:.1%} | colisiones: {collision_rate(codes3):.2%} | "
      f"pureza: {purity(codes3[:, 0], cats).mean():.3f} vs aleatorio {purity(np.random.default_rng(0).permutation(codes3[:, 0]), cats).mean():.3f}")
sid_codes = add_dedup_token(codes3)
item_tokens, level_sizes, offsets, VOCAB = build_token_tables(sid_codes, CODEBOOK_SIZE)
trie = SemanticIDTrie(sid_codes)

In [ ]:
# Cold start: retiramos el 5 % de los ítems del entrenamiento del modelo generativo
H_HIST = 20
rng = np.random.default_rng(seed)
cold = set(rng.choice(np.arange(1, gn_items + 1), max(1, gn_items // 20), replace=False).tolist())
warm_train = [[i for i in s if i not in cold] for s in g_train]
tr_hist, tr_tgt = gen_examples(warm_train, H=H_HIST, max_examples=50_000 if FAST_DEV_RUN else 2_000_000)
torch.manual_seed(seed)
tiger = TigerMini(VOCAB, level_sizes, offsets, d=128, nhead=4, n_layers=4, ff=1024, dropout=0.1, max_src=H_HIST * 4)
vsel = rng.choice(len(g_valid[0]), min(2000, len(g_valid[0])), replace=False)
val_small = ([g_valid[0][i] for i in vsel], [g_valid[1][i] for i in vsel])
log = train_tiger(tiger, tr_hist, tr_tgt, item_tokens, epochs=2 if FAST_DEV_RUN else 20, batch_size=512, lr=5e-4, H=H_HIST,
                  eval_fn=lambda m: eval_generative(m, *val_small, trie, item_tokens, beam=10, ks=(10,), H=H_HIST))

In [ ]:
N_EVAL = min(len(g_test[0]), 1000 if FAST_DEV_RUN else 20000)
sel = rng.choice(len(g_test[0]), N_EVAL, replace=False)
tH, tT = [g_test[0][i] for i in sel], [g_test[1][i] for i in sel]
res = {"TIGER-mini (trie)": eval_generative(tiger, tH, tT, trie, item_tokens, beam=20, H=H_HIST),
       "TIGER-mini (sin trie)": eval_generative(tiger, tH, tT, trie, item_tokens, beam=20, H=H_HIST, constrained=False)}
popg = np.bincount(np.concatenate(g_train), minlength=gn_items + 1).astype(np.float32)
res["Popularidad"] = {k.replace("HR@", "Recall@"): v for k, v in
                      evaluate_next_item(lambda H: torch.from_numpy(np.tile(popg, (len(H), 1))), tH, tT, ks=(10, 20)).items()}
cold_idx = [k for k, t in enumerate(g_test[1]) if t in cold]
if cold_idx:
    res["TIGER-mini · sólo objetivos cold"] = eval_generative(tiger, [g_test[0][k] for k in cold_idx], [g_test[1][k] for k in cold_idx],
                                                              trie, item_tokens, beam=20, H=H_HIST)
display(pd.DataFrame(res).T.round(4))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(log["epoch"], log["loss"], "o-"); ax[0].set(title="Pérdida TIGER-mini", xlabel="época")
pd.DataFrame(res).T["Recall@10"].plot.barh(ax=ax[1], color="#6366f1"); ax[1].set(title="Recall@10 (test)")
plt.tight_layout(); plt.show()

**Lectura esperada.** El trie lleva los IDs inválidos a 0 %. El Recall@10 generativo debería superar con holgura a popularidad; frente a SASRec-CE (lección) puede quedar por debajo en ítems populares.
Lo interesante para negocio es la fila *cold*: los ítems que el generador **nunca vio como objetivo** siguen siendo recuperables porque su SID comparte prefijos con ítems parecidos
— algo imposible para un modelo con tabla de IDs.

## 🚀 Retos extra (nivel experto)
1. **RQ-KMeans vs RQ-VAE** como tokenizador del mismo modelo: ¿qué tokenizador da mejor Recall@10 y mejor *cold start*?
2. **Señal colaborativa en el tokenizador**: concatena al embedding de texto el embedding de ítem de un SASRec/MF entrenado (normalizado) antes del RQ-VAE (idea de SID-v2/OneRec). ¿Mejora la pureza *conductual*?
3. **Token de usuario** (TIGER hashea el ID de usuario en 2.000 tokens) y **features de contexto** en el encoder.
4. **Híbrido LIGER**: usa los top-50 del beam como candidatos y re-puntúalos con el producto escalar de SASRec.
5. **Filtro de negocio en el trie**: excluye en decodificación los productos «sin stock» (máscara aleatoria del 20 %) y compara con filtrar *después* del beam.
6. **Escala**: `Books` o `Clothing_Shoes_and_Jewelry` (cientos de miles de ítems) en A100: ¿cómo crecen colisiones, tamaño del trie y latencia del beam?

## 🤔 Reflexión (producción / MLOps)
- Si reentrenas el RQ-VAE cada mes, todos los SIDs cambian: ¿cómo versionas tokenizador + trie + modelo y haces *rollback* (módulo 17)?
- ¿Qué latencia p99 tiene tu beam search para beam = 20 y cómo la reducirías (KV-cache, multi-token, *distillation* a two-tower)?
- ¿Qué métricas de ecosistema vigilarías al desplegar (exposición de productos nuevos, diversidad, módulo 13)?